In [1]:
%%writefile state/schema.py
from typing import TypedDict, Annotated, List, Dict, Optional
from operator import add


class ResearchState(TypedDict):
    """Shared state that flows through every node in the research graph. (v2, Day 7)"""
    query: str
    plan: List[str]

    
    next_agent: str
    iteration_count: int          
    needs_more_research: bool

   
    search_results: Annotated[List[Dict], add]
    extracted_facts: Annotated[List[Dict], add]
    fact_check_results: Annotated[List[Dict], add]
    summaries: Annotated[List[Dict], add]

 
    errors: Annotated[List[Dict], add]

    final_report: Optional[str]

    model_usage_log: Annotated[List[Dict], add]

Overwriting state/schema.py


In [ ]:
import os, sys, importlib
sys.path.append(".")

checks = []


for folder in ["agents", "tools", "graph", "state", "tests", "data", "docs"]:
    checks.append((f"folder {folder}/", os.path.isdir(folder)))

for path in ["README.md", "requirements.txt", ".gitignore", "docs/architecture.md",
             "state/schema.py", "tools/llm_clients.py", "tools/search_tool.py", "tools/logger.py"]:
    checks.append((f"file {path}", os.path.isfile(path)))


with open("docs/architecture.md", encoding="utf-8") as f:
    arch = f.read()
for section in ["Supervisor pattern", "Hierarchical", "Network"]:
    checks.append((f"architecture.md mentions '{section}'", section in arch))

from state.schema import ResearchState
for field in ["next_agent", "iteration_count", "errors"]:
    checks.append((f"schema has '{field}'", field in ResearchState.__annotations__))

try:
    from tools.llm_clients import reasoning_llm
    ok = "ok" in reasoning_llm.invoke("Reply with exactly: ok").content.lower()
    checks.append(("reasoning_llm responds", ok))
except Exception as e:
    checks.append((f"reasoning_llm responds ({type(e).__name__})", False))

for name, passed in checks:
    print("✅" if passed else "❌", name)

print(f"\n{sum(p for _, p in checks)}/{len(checks)} checks passed")